# Goal: statistical-summary 

In [1]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
Date: 10-10-2026 14:38 IST
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\nDate: 10-10-2026 14:38 IST\n'

In [2]:
import polars as pl

In [3]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-06\data\lev-06_merged.parquet"
pdf = pl.scan_parquet(path)

In [4]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('Item_Code', String),
        ('Total_Consumption_Quantity', String),
        ('Total_Consumption_Value', String),
        ('Source', String),
        ('Multiplier', Int64)])

# Useful Variables

In [5]:
lev_06 = [
    'Total_Consumption_Quantity',
    'Total_Consumption_Value',
    'Source',
    'Multiplier',
]



In [6]:
df = pdf.select(lev_06)

In [7]:
df.head(2).collect()

Total_Consumption_Quantity,Total_Consumption_Value,Source,Multiplier
str,str,str,i64
"""3""","""160""","""""",192686
"""""","""200""","""1""",192686


In [8]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in lev_06]
)

In [9]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

Total_Consumption_Quantity,Total_Consumption_Value,Source,Multiplier
u32,u32,u32,u32
138,1939,5,23546


# Logic

In [10]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_5552\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [11]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
Total_Consumption_Quantity,202902.0,3311626.0,20.164759,47.165606,1.0,2.0,5.0,12.0,950.0
Total_Consumption_Value,3514528.0,0.0,123.376579,208.644401,1.0,30.0,60.0,135.0,35150.0
Multiplier,3514528.0,0.0,109151.107262,79679.410344,369.0,51676.0,112163.0,149240.0,2366902.0


# Categorical Columns

In [12]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))

Source


Source,count
i32,u32
1,2342006
null,1159480
9,6376
6,4816
5,1850
